<a href="https://colab.research.google.com/github/mmatinm/SmartCane-AI-Visual-Navigation-System/blob/main/training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Function

In [2]:
import os
import yaml
import random
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

def report_yolo_dataset(dataset_dir, show_examples=True, num_examples=8):
    def count_images(split):
        img_dir = os.path.join(dataset_dir, split, "images")
        if os.path.exists(img_dir):
            return len([f for f in os.listdir(img_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
        else:
            return 0

    def analyze_split_labels(split, num_classes):
        label_dir = os.path.join(dataset_dir, split, "labels")
        class_image_counts = [0] * num_classes
        overlap_count = 0
        empty_or_missing_count = 0

        image_dir = os.path.join(dataset_dir, split, "images")
        if not os.path.exists(image_dir):
            return class_image_counts, overlap_count, empty_or_missing_count

        image_files = [f for f in os.listdir(image_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

        for img_file in image_files:
            img_stem = os.path.splitext(img_file)[0]
            label_path = os.path.join(label_dir, img_stem + ".txt")
            classes_in_image = set()

            if os.path.exists(label_path):
                with open(label_path, 'r') as f:
                    lines = f.readlines()
                    for line in lines:
                        parts = line.strip().split()
                        if len(parts) >= 1:
                            cls_id = int(float(parts[0]))
                            if 0 <= cls_id < num_classes:
                                classes_in_image.add(cls_id)
            else:
                empty_or_missing_count += 1
                continue  # no label file

            if len(classes_in_image) == 0:
                empty_or_missing_count += 1
            else:
                if len(classes_in_image) > 1:
                    overlap_count += 1
                for cls_id in classes_in_image:
                    class_image_counts[cls_id] += 1

        return class_image_counts, overlap_count, empty_or_missing_count

    # Print image counts
    train_count = count_images("train")
    valid_count = count_images("valid")
    test_count = count_images("test")

    print(f"📊 Image counts:")
    print(f"  Train: {train_count}")
    print(f"  Validation: {valid_count}")
    print(f"  Test: {test_count}")

    # Load class names
    data_yaml_path = os.path.join(dataset_dir, "data.yaml")
    with open(data_yaml_path, 'r') as f:
        data_yaml = yaml.safe_load(f)

    class_names = data_yaml.get("names", [])
    num_classes = len(class_names)

    print("\n🏷️ Classes in dataset:")
    for i, cls in enumerate(class_names):
        print(f"  {i}: {cls}")

    # Count per-class image occurrences and extra stats
    print("\n📂 Per-class image counts (number of images where each class appears):")
    for split in ["train", "valid", "test"]:
        class_image_counts, overlap_count, empty_count = analyze_split_labels(split, num_classes)
        print(f"  {split.capitalize()}:")
        for i, count in enumerate(class_image_counts):
            print(f"    {class_names[i]} ({i}): {count}")
        print(f"    ➕ Overlap images (contain ≥2 classes): {overlap_count}")
        print(f"    🚫 Images without any class (empty or missing labels): {empty_count}")

    if not show_examples:
        return

    # Pick examples from train split
    image_dir = os.path.join(dataset_dir, "train", "images")
    label_dir = os.path.join(dataset_dir, "train", "labels")

    image_files = [f for f in os.listdir(image_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    if len(image_files) == 0:
        print("\n⚠️ No images found in train/images to display.")
        return

    sample_files = random.sample(image_files, min(num_examples, len(image_files)))

    print(f"\n🖼️ Showing {len(sample_files)} random annotated images...")

    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    axes = axes.flatten()

    for ax, fname in zip(axes, sample_files):
        img_path = os.path.join(image_dir, fname)
        label_path = os.path.join(label_dir, os.path.splitext(fname)[0] + ".txt")

        image = Image.open(img_path)
        w, h = image.size
        ax.imshow(image)
        ax.axis('off')

        if os.path.exists(label_path):
            with open(label_path, "r") as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) != 5:
                        continue
                    cls_id, xc, yc, bw, bh = map(float, parts)
                    x = (xc - bw / 2) * w
                    y = (yc - bh / 2) * h
                    box_w = bw * w
                    box_h = bh * h
                    rect = patches.Rectangle((x, y), box_w, box_h,
                                             linewidth=2, edgecolor='red', facecolor='none')
                    ax.add_patch(rect)
                    if class_names:
                        ax.text(x, y, class_names[int(cls_id)], color='yellow', fontsize=10,
                                bbox=dict(facecolor='black', alpha=0.5))

    plt.tight_layout()
    plt.show()


# Preprocessing

In [ ]:
import os
import shutil
import yaml
from pathlib import Path

# Input datasets
datasets = [
    "/content/drive/MyDrive/SmartCane/final_bus",
    "/content/drive/MyDrive/SmartCane/final_crosswalk",
    "/content/drive/MyDrive/SmartCane/final_escalator",
    "/content/drive/MyDrive/SmartCane/final_rglight",
    "/content/drive/MyDrive/SmartCane/final_safety"
]

# Output path
merged_dataset_path = "/content/drive/MyDrive/SmartCane/merged_dataset"
splits = ["train", "valid", "test"]

# Create merged dataset folders
for split in splits:
    for subdir in ["images", "labels"]:
        os.makedirs(os.path.join(merged_dataset_path, split, subdir), exist_ok=True)

# Merge class names and keep mapping
class_name_to_index = {}
all_classes = []
next_class_index = 0

def load_yaml_classes(path):
    with open(path, "r") as f:
        data = yaml.safe_load(f)
    return data["names"]

def remap_labels(label_path, class_mapping):
    with open(label_path, "r") as f:
        lines = f.readlines()

    new_lines = []
    for line in lines:
        if line.strip() == "":
            continue
        parts = line.strip().split()
        old_class = int(parts[0])
        new_class = class_mapping[old_class]
        parts[0] = str(new_class)
        new_lines.append(" ".join(parts))

    return new_lines

# Process each dataset
for dataset in datasets:
    dataset_name = Path(dataset).name

    # Load class names from data.yaml
    data_yaml_path = os.path.join(dataset, "data.yaml")
    class_names = load_yaml_classes(data_yaml_path)

    # Map local class indices to global ones
    local_to_global = {}
    for i, name in enumerate(class_names):
        if name not in class_name_to_index:
            class_name_to_index[name] = next_class_index
            all_classes.append(name)
            next_class_index += 1
        local_to_global[i] = class_name_to_index[name]

    # Go through each split
    for split in splits:
        image_dir = os.path.join(dataset, split, "images")
        label_dir = os.path.join(dataset, split, "labels")

        if not os.path.exists(image_dir):
            continue  # Skip if split not present

        for image_file in os.listdir(image_dir):
            if not image_file.lower().endswith((".jpg", ".jpeg", ".png")):
                continue

            base_name = f"{dataset_name}_{image_file}"
            name_wo_ext = os.path.splitext(base_name)[0]

            # Copy image
            src_img = os.path.join(image_dir, image_file)
            dst_img = os.path.join(merged_dataset_path, split, "images", base_name)
            shutil.copy2(src_img, dst_img)

            # Copy & remap label
            label_filename = os.path.splitext(image_file)[0] + ".txt"
            src_label = os.path.join(label_dir, label_filename)
            dst_label = os.path.join(merged_dataset_path, split, "labels", name_wo_ext + ".txt")

            if os.path.exists(src_label):
                new_label_lines = remap_labels(src_label, local_to_global)
                with open(dst_label, "w") as f:
                    f.write("\n".join(new_label_lines) + "\n")

# Save final data.yaml
final_yaml = {
    "train": os.path.join(merged_dataset_path, "train"),
    "val": os.path.join(merged_dataset_path, "valid"),
    "test": os.path.join(merged_dataset_path, "test"),
    "nc": len(all_classes),
    "names": all_classes
}

with open(os.path.join(merged_dataset_path, "data.yaml"), "w") as f:
    yaml.dump(final_yaml, f)

print("✅ Merged YOLOv8 dataset saved at:", merged_dataset_path)


In [ ]:
report_yolo_dataset('/content/drive/MyDrive/SmartCane/final10')